In [1]:
import xarray as xr
from dask.distributed import Client
from dask_jobqueue import SLURMCluster

# 1. Define your Slurm worker requirements
# This is like writing a small #SBATCH script inside Python
cluster = SLURMCluster(
    queue="geocean",  # Your partition name
    cores=1,  # CPUs per Slurm job
    memory="8GB",  # RAM per Slurm job
    walltime="01:00:00",  # Time limit per job
    job_script_prologue=[  # Essential for Conda envs
        "source /nfs/home/geocean/javiert/miniforge3/etc/profile.d/conda.sh",
        "conda activate bluemath-312",
    ],
    # Important: local_directory should be on high-speed scratch disk
    local_directory="/tmp/",
)

# 2. Scale the cluster (e.g., ask Slurm for 5 jobs = 20 cores/80GB RAM total)
cluster.scale(jobs=10)
client = Client(cluster)

# 3. Open the Dashboard link printed here to monitor the RAM!
print(client.dashboard_link)

http://192.168.202.87:8787/status


In [ ]:
# Use h5netcdf engine for faster HPC performance
ds = xr.open_mfdataset(
    "/nfs/home/geocean/gutierrezd/netherlands/*.nc",
    engine="h5netcdf",
    parallel=True,
)

# Rechunking logic:
# Since you want "one seapoint for all time,"
# you want to increase the time chunk size before writing to Zarr.
ds = ds.chunk({"time": -1, "seapoint": 26})

# Write to Zarr using the cluster
ds.to_zarr("inputs/WHACS/processed/Netherlands_WHACS_seapoint.zarr", consolidated=True)

/tmp/ipykernel_2972979/3124808366.py:2: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'time' ('time',) The recommendation is to set join explicitly for this case.
  ds = xr.open_mfdataset(
/nfs/home/geocean/javiert/miniforge3/envs/bluemath-312/lib/python3.12/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/nfs/home/geocean/javiert/miniforge3/envs/bluemath-312/lib/python3.12/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 10
  result = blockwise(
/nfs/home/geocean/javiert/miniforge3/envs/bluemath-312/lib/python3.12/site-packages/dask/array/core.py:5201: PerformanceWarning: Increasing number of chunks by factor of 10
  resul

In [ ]:
import xarray as xr

output = xr.open_zarr("inputs/WHACS/processed/Netherlands_WHACS_seapoint.zarr")
output

<xarray.Dataset> Size: 1GB
Dimensions:                      (time: 394464, seapoint: 10)
Coordinates:
  * time                         (time) datetime64[ns] 3MB 1979-01-01 ... 202...
  * seapoint                     (seapoint) int32 40B 2323617 ... 2326141
Data variables: (12/41)
    cge                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    dir                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    dp                           (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    fp                           (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    hs                           (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    latitude                     (time, seapoint) float32 16MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    ...                           ...
    ptp3                         (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    spr                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    t01                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    t02                          (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    uwnd                         (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
    vwnd                         (time, seapoint) float64 32MB dask.array<chunksize=(394464, 10), meta=np.ndarray>
Attributes: (12/59)
    Conventions:                     CF-1.11 ACDD-1.3
    title:                           Wave Hindcast for the Australian Climate...
    summary:                         Wave Hindcast for the Australian Climate...
    keywords:                        WHACS, ACS, waves, ocean, coast, WWIII, ...
    id:                              BoM-CSIRO-WHACS-001
    naming_authority:                au.csiro
    ...                              ...
    geospatial_lon_max:              179.9375
    geospatial_vertical_positive:    up
    time_coverage_start:             1979-01-01T00:00:00
    time_coverage_end:               1979-01-31T23:00:00
    time_coverage_duration:          P30DT23H0M0S
    time_coverage_resolution:        P0DT1H0M0S